# Maia Corpus v8 — Production Resume v2 + Wikimedia Rights Review + Provenance Audit

Recovery-safe continuation with incremental metadata/provenance review. **No automatic license approval.**


In [ ]:
# 1. Mount Drive (Colab only)
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; use locally mounted paths.')

In [ ]:
# 2. Dependencies and environment
import sys, subprocess
try:
    import transformers, tokenizers, numpy
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.45,<5', 'tokenizers>=0.20', 'numpy>=1.26'])
import os, json, gzip, hashlib, sqlite3, time, shutil, platform, random, collections, struct
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
from transformers import AutoTokenizer
print('Python:', platform.python_version())
print('NumPy:', np.__version__)
print('Transformers:', __import__('transformers').__version__)

In [ ]:
# Configuration — original production paths and parameters
ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001')
OUT = ROOT / 'pretraining_production_v1_full'
FREEZE_MANIFEST = ROOT / 'freeze_v8_001/manifest.json'
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
RECOVERY = OUT / 'recovery_audit_v4'
SEED_DB = RECOVERY / 'state_operational_v5.sqlite3'
WORK_DIR = RECOVERY / 'production_resume_v2'
DB = WORK_DIR / 'state_resume_v2.sqlite3'
CONFIG_PATH = OUT / 'run_config.json'
EXPECTED_TOTAL_CHUNKS = 21021
BLOCK_SIZE = 2048
SHARD_MAX_BLOCKS = 512
SPLIT_PERCENT = {'train':94,'validation':3,'test':3}
SPLIT_SEED = 'maia-v8-split-v1'
EOS_PER_DOCUMENT = True
SOURCE_NAMES = ['fineweb2_es','fineweb2_pt','fineweb_edu_en','wikipedia_en','wikipedia_es','wikipedia_pt']
LANGUAGE = {'fineweb2_es':'es','fineweb2_pt':'pt','fineweb_edu_en':'en','wikipedia_en':'en','wikipedia_es':'es','wikipedia_pt':'pt'}
LANGUAGE_TARGET = {'pt':0.40,'en':0.35,'es':0.25}
TARGET_TRAIN_EXPOSURES = 10_100_000_000
PRODUCTION_MODE = True
MAX_NEW_CHUNKS_PER_SESSION = 100  # Change to None for all remaining chunks in one session
PROGRESS_EVERY_NEW_CHUNKS = 10
VERIFY_EVERY_NEW_CHUNKS = 50
# No processing until both switches are deliberately set below.
assert OUT.is_dir() and SEED_DB.is_file() and CONFIG_PATH.is_file()
print('Seed:', SEED_DB)
print('Working DB:', DB)
print('Original DB untouched:', OUT/'state.sqlite3')

In [ ]:
# 4. Validate source freeze and tokenizer fingerprint
assert FREEZE_MANIFEST.is_file(), f'Missing freeze: {FREEZE_MANIFEST}'
freeze_bytes = FREEZE_MANIFEST.read_bytes()
freeze = json.loads(freeze_bytes)
freeze_sha = hashlib.sha256(freeze_bytes).hexdigest()
assert TOKENIZER_DIR.is_dir(), f'Missing tokenizer: {TOKENIZER_DIR}'
TOK_FILES = ['merges.txt','tokenizer_config.json','vocab.json','tokenizer.json']
for filename in TOK_FILES:
    assert (TOKENIZER_DIR/filename).is_file(), f'Missing tokenizer file: {filename}'
file_hashes = {f:hashlib.sha256((TOKENIZER_DIR/f).read_bytes()).hexdigest() for f in TOK_FILES}
# These hashes are taken from the successful stratified pilot v2.
EXPECTED_HASHES = {
 'merges.txt':'b4b5c08bd27684fb3c6426ffe33adaaf7375340c733dad2f1d609d90767cef8a',
 'tokenizer_config.json':'a1cf74974f6d8a98aef2b7a2bb34084a6ad9d79c4e36e7280d369628149c7062',
 'vocab.json':'63b18c80a572b14e90de3529ac068b76ca49e449b7c4219eccbc05d6cee4cf6c',
 'tokenizer.json':'fd6cc1c4c1a487c5c4b45ee812762c471e5776ff1845fa48ad04c245ae4644a6'
}
assert file_hashes == EXPECTED_HASHES, 'Tokenizer files differ from validated pilot v2.'
tok = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR), local_files_only=True, use_fast=True)
assert len(tok) == 50257, f'Unexpected vocab size: {len(tok)}'
assert tok.eos_token_id == 2, f'Unexpected EOS: {tok.eos_token_id}'
assert tok.pad_token_id == 1, f'Unexpected PAD: {tok.pad_token_id}'
print('Freeze manifest SHA-256:', freeze_sha)
print('Tokenizer file hashes: MATCH pilot v2')
print('Historical tokenizer identity: NOT PROVEN')

In [ ]:
# Load the EXISTING immutable inventory, rather than rescan or recreate it.
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert config['production_mode'] is True
assert config['schema_version'] == '1.1-full'
assert config['block_size'] == BLOCK_SIZE
assert config['shard_max_blocks'] == SHARD_MAX_BLOCKS
assert config['split_percent'] == SPLIT_PERCENT
assert config['split_seed'] == SPLIT_SEED
assert config['eos_per_document'] == EOS_PER_DOCUMENT
assert config['tokenizer_file_sha256'] == file_hashes
assert config['freeze_manifest_sha256'] == freeze_sha
assert config['source_names'] == SOURCE_NAMES
INVENTORY_BY_PATH = {x['path']:x for x in config['selected_inventory']}
assert len(INVENTORY_BY_PATH) == EXPECTED_TOTAL_CHUNKS
selected = {s:[] for s in SOURCE_NAMES}
for record in config['selected_inventory']:
    selected[record['source']].append(Path(record['path']))
for s in SOURCE_NAMES:
    selected[s].sort(key=str)
selected_paths = [p for s in SOURCE_NAMES for p in selected[s]]
config_digest=hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
print('Inventory:', len(selected_paths), 'chunks; configuration SHA:', config_digest)

In [ ]:
# Initialize operational copy ONCE; subsequent sessions reuse it.
# Never copy the seed over an existing working database.
WORK_DIR.mkdir(parents=True,exist_ok=True)
if not DB.exists():
    assert not any(WORK_DIR.glob('state_resume_v2.sqlite3*')), 'Unexpected working SQLite sidecar: inspect manually'
    source=sqlite3.connect(SEED_DB.resolve().as_uri()+'?mode=ro',uri=True)
    try:
        assert source.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
        assert source.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0]==20371
        assert source.execute('SELECT COUNT(*) FROM seen').fetchone()[0]==9788069
        assert source.execute('SELECT COUNT(*) FROM shard').fetchone()[0]==60612
        temp=WORK_DIR/'state_resume_v2.sqlite3.initializing'
        assert not temp.exists(), 'Incomplete previous initialization; inspect manually'
        dest=sqlite3.connect(temp)
        try: source.backup(dest); dest.commit()
        finally: dest.close()
        check=sqlite3.connect(temp)
        try: assert check.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
        finally: check.close()
        os.replace(temp,DB)
    finally: source.close()
    print('Created separate working DB from audited v5.')
else:
    print('Resuming existing working DB; seed NOT recopied.')
con=sqlite3.connect(DB,timeout=120)
con.execute('PRAGMA journal_mode=DELETE')
con.execute('PRAGMA synchronous=FULL')
con.execute('PRAGMA busy_timeout=120000')
assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok'
assert con.execute('SELECT COUNT(*) FROM recovery_provenance').fetchone()[0]==306
completed_paths={p for (p,) in con.execute('SELECT path FROM completed_chunk')}
assert completed_paths.issubset(INVENTORY_BY_PATH)
assert len(completed_paths)<=EXPECTED_TOTAL_CHUNKS
print('Completed:',len(completed_paths),'remaining:',EXPECTED_TOTAL_CHUNKS-len(completed_paths))

In [ ]:
# Original deterministic helpers; no expensive full-shard re-audit.
import unicodedata

def canonical_text(value):
    return unicodedata.normalize('NFC',value).strip()

def document_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def split_for_hash(h):
    bucket = int.from_bytes(hashlib.sha256((SPLIT_SEED + ':' + h).encode()).digest()[:8],'big') % 10000
    if bucket < SPLIT_PERCENT['train']*100: return 'train'
    if bucket < (SPLIT_PERCENT['train']+SPLIT_PERCENT['validation'])*100: return 'validation'
    return 'test'

def shard_file(source,split,index):
    return OUT/'shards'/source/split/f'{source}_{split}_{index:06d}.bin'


In [ ]:
# 8. Process a chunk atomically, retaining tails between chunks
# Shards use little-endian uint32 token IDs, exactly BLOCK_SIZE IDs per block.
# Each (source,split) has its own token stream; streams never mix languages/splits.
# IMPORTANT: This cell only defines functions; the next cell performs processing.

def process_chunk(source,path):
    path = Path(path)
    row=con.execute('SELECT 1 FROM completed_chunk WHERE path=?',(str(path),)).fetchone()
    if row: return 'SKIP_COMPLETED'
    # Check source metadata against the immutable run config.
    record = INVENTORY_BY_PATH[str(path)]
    st=path.stat()
    assert (st.st_size,st.st_mtime_ns)==(record['size'],record['mtime_ns']), f'Source changed: {path}'
    # Prepare pending stream state in RAM. Existing SQLite state is not changed until commit.
    buffers={}
    for split in SPLIT_PERCENT:
        row=con.execute('SELECT ids FROM pending WHERE source=? AND split=?',(source,split)).fetchone()
        buffers[split]=np.frombuffer(row[0],dtype='<u4').astype('<u4').tolist() if row else []
    new_hashes=[]; local_hashes=set(); docs=dups=tokens=0
    new_shards=[]
    try:
        with gzip.open(path,'rt',encoding='utf-8') as stream:
            for line in stream:
                if not line.strip(): continue
                obj=json.loads(line)
                raw=obj.get('text')
                if not isinstance(raw,str): raise ValueError(f'Missing text in {path}')
                txt=canonical_text(raw)
                if not txt: continue
                h=document_hash(txt)
                if h in local_hashes or con.execute('SELECT 1 FROM seen WHERE hash=?',(h,)).fetchone():
                    dups+=1; continue
                local_hashes.add(h)
                split=split_for_hash(h)
                ids=tok.encode(txt,add_special_tokens=False)
                if EOS_PER_DOCUMENT: ids.append(tok.eos_token_id)
                assert all(0 <= t < len(tok) for t in ids)
                buffers[split].extend(ids)
                new_hashes.append((h,split)); docs+=1; tokens+=len(ids)
                if not PRODUCTION_MODE and docs+dups>=MAX_DOCS_PER_CHUNK_PILOT: break
        # Prepare full blocks for each split, and preserve incomplete tails.
        for split in SPLIT_PERCENT:
            buf=buffers[split]
            nfull=len(buf)//BLOCK_SIZE
            if nfull:
                previous=con.execute('SELECT COALESCE(MAX(shard_index),-1) FROM shard WHERE source=? AND split=?',(source,split)).fetchone()[0]
                for start_block in range(0,nfull,SHARD_MAX_BLOCKS):
                    take=min(SHARD_MAX_BLOCKS,nfull-start_block)
                    index=previous+1+sum(1 for x in new_shards if x[0]==source and x[1]==split)
                    target=shard_file(source,split,index)
                    assert not target.exists(), f'Uncommitted/orphan shard exists: {target}; investigate before resuming.'
                    target.parent.mkdir(parents=True,exist_ok=True)
                    arr=np.asarray(buf[start_block*BLOCK_SIZE:(start_block+take)*BLOCK_SIZE],dtype='<u4')
                    payload=arr.tobytes()
                    temp=target.with_suffix('.bin.tmp')
                    assert not temp.exists(), f'Orphan temporary shard: {temp}'
                    with temp.open('xb') as f:
                        f.write(payload); f.flush(); os.fsync(f.fileno())
                    os.replace(temp,target)
                    new_shards.append((source,split,index,str(target),take,hashlib.sha256(payload).hexdigest()))
            buffers[split]=buf[nfull*BLOCK_SIZE:]
            assert len(buffers[split])<BLOCK_SIZE
        with con:
            con.executemany('INSERT INTO seen(hash,split) VALUES(?,?)',new_hashes)
            con.executemany('INSERT INTO shard(source,split,shard_index,path,blocks,sha256) VALUES(?,?,?,?,?,?)',new_shards)
            for split in SPLIT_PERCENT:
                tail=np.asarray(buffers[split],dtype='<u4').tobytes()
                con.execute('INSERT INTO pending(source,split,ids) VALUES(?,?,?) ON CONFLICT(source,split) DO UPDATE SET ids=excluded.ids',(source,split,tail))
            con.execute('INSERT INTO completed_chunk(path,source,docs,duplicates,tokens,completed_utc) VALUES(?,?,?,?,?,?)',
                        (str(path),source,docs,dups,tokens,datetime.now(timezone.utc).isoformat()))
        return f'OK docs={docs} dup={dups} tokens={tokens} shards={len(new_shards)}'
    except Exception:
        # Never silently remove generated files: inspect orphans on recovery.
        raise

In [ ]:
# Fast safety gate. No automatic quarantine or file movement.
committed = {str(Path(p).resolve()) for (p,) in con.execute('SELECT path FROM shard')}
shards_root=OUT/'shards'
orphans=[]
if shards_root.exists():
    for p in shards_root.rglob('*'):
        if p.is_file() and (p.name.endswith('.bin') or p.name.endswith('.bin.tmp')):
            if str(p.resolve()) not in committed: orphans.append(p)
assert not orphans, 'Orphan shards detected: inspect before continuing: '+str(orphans[:10])
assert con.execute('SELECT COUNT(*) FROM shard').fetchone()[0] >= 60612
for s in SOURCE_NAMES:
    for split in SPLIT_PERCENT:
        maximum=con.execute('SELECT COALESCE(MAX(shard_index),-1) FROM shard WHERE source=? AND split=?',(s,split)).fetchone()[0]
        target=shard_file(s,split,maximum+1)
        assert not target.exists(), 'Next shard output already exists: '+str(target)
print('FAST PREFLIGHT: PASS; no orphan shards; no output collisions')

## Wikimedia Rights Review — Wikipedia PT

Collect a source-level evidence dossier and a **draft** attribution notice for the 650 remaining chunks. This step samples metadata only; it does **not** determine that every article is covered by CC BY-SA 4.0, clear third-party rights, or authorize training. It preserves existing human decisions and keeps the production gate closed until the separate license audit and independent reviewer approval succeed.


In [ ]:
# Wikimedia Rights Review — evidence dossier only; NO automatic approval
# Run after FAST PREFLIGHT and before License Compliance Audit.
import json, hashlib, gzip
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

RIGHTS_DIR = WORK_DIR / 'wikimedia_rights_review'
RIGHTS_DIR.mkdir(parents=True, exist_ok=True)
EVIDENCE_PATH = RIGHTS_DIR / 'wikipedia_pt_evidence.json'
NOTICE_PATH = RIGHTS_DIR / 'NOTICE_WIKIPEDIA_PT_DRAFT.md'
REGISTRY_PATH = WORK_DIR / 'license_registry.json'

INVENTORY_SHA256 = hashlib.sha256(json.dumps(
    config['selected_inventory'], sort_keys=True, ensure_ascii=False,
    separators=(',', ':')
).encode('utf-8')).hexdigest()

wiki_rows = [r for r in config['selected_inventory'] if r['source'] == 'wikipedia_pt']
wiki_remaining = [r for r in wiki_rows if r['path'] not in completed_paths]
assert len(wiki_remaining) == 650, ('Unexpected remaining Wikipedia chunks', len(wiki_remaining))

# Inspect only a small deterministic sample of gzip JSONL metadata.
# This is not a proof that all articles have identical licensing.
sample_paths = [wiki_remaining[i]['path'] for i in sorted(set([
    0, len(wiki_remaining)//4, len(wiki_remaining)//2,
    3*len(wiki_remaining)//4, len(wiki_remaining)-1
]))]
sample_fields = Counter()
sample_examples = []
for filename in sample_paths:
    path = Path(filename)
    assert path.is_file(), path
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for _ in range(3):
            line = f.readline()
            if not line:
                break
            record = json.loads(line)
            assert isinstance(record, dict)
            sample_fields.update(record.keys())
            sample_examples.append({
                'chunk': path.name,
                'available_fields': sorted(record.keys()),
                'source_reference': {
                    k: str(record[k])[:500] for k in
                    ('url','source_url','title','id','revision_id','license','timestamp','date')
                    if k in record and record[k] is not None
                }
            })

# References are authoritative starting points, not proof of the
# specific dataset snapshot or absence of third-party material.
references = [
    {'name':'Wikimedia Foundation Terms of Use',
     'url':'https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use'},
    {'name':'CC BY-SA 4.0 legal code',
     'url':'https://creativecommons.org/licenses/by-sa/4.0/legalcode'},
    {'name':'Wikimedia content reuse guidance',
     'url':'https://www.mediawiki.org/wiki/Wikimedia_APIs/Content_reuse'},
    {'name':'Creative Commons — Using CC-licensed Works for AI Training',
     'url':'https://creativecommons.org/using-cc-licensed-works-for-ai-training-2/'},
]

questions = [
    'Identify the exact Wikipedia Portuguese dataset, snapshot date, extraction method and original URLs.',
    'Confirm text-only extraction and treatment of quoted, imported and third-party content.',
    'Confirm attribution mechanism (article links/history and contributors where required).',
    'Review CC BY-SA obligations and applicable law for training, public weights, outputs and redistribution.',
    'Confirm rights scope includes commercial use and redistribution of Maia model weights.',
    'Determine whether a per-article attribution manifest is needed and feasible.',
    'Review license obligations for already-processed Wikipedia and FineWeb sources separately.',
]

report = {
    'schema_version': 1,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'REVIEW_REQUIRED',
    'source': 'wikipedia_pt',
    'inventory_sha256': INVENTORY_SHA256,
    'total_source_chunks': len(wiki_rows),
    'remaining_source_chunks': len(wiki_remaining),
    'sampled_chunk_files': [Path(x).name for x in sample_paths],
    'sampled_records': len(sample_examples),
    'sample_metadata_field_frequency': dict(sample_fields),
    'sample_references': sample_examples,
    'candidate_license': 'CC BY-SA 4.0 (verify applicability to exact snapshot and each exception)',
    'intended_uses': ['model_training','commercial_use','public_weight_distribution','redistribution'],
    'reference_urls': references,
    'outstanding_questions': questions,
    'snapshot_verified': False,
    'third_party_exceptions_cleared': False,
    'human_rights_approval': False,
    'note': 'Automated evidence collection only; does not authorize processing.'
}

# Never overwrite an existing evidence file silently.
if EVIDENCE_PATH.exists():
    existing = json.loads(EVIDENCE_PATH.read_text(encoding='utf-8'))
    assert existing.get('inventory_sha256') == INVENTORY_SHA256, 'Existing evidence belongs to another inventory'
    print('Existing evidence preserved:', EVIDENCE_PATH)
else:
    with EVIDENCE_PATH.open('x', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print('Created evidence:', EVIDENCE_PATH)

notice = f'''# DRAFT — Wikipedia Portuguese training-data attribution

**Status: DRAFT — NOT A LEGAL CLEARANCE OR FINAL LICENSE NOTICE.**

Maia Corpus v8 includes text associated with Wikipedia in Portuguese.
Source: https://pt.wikipedia.org/

The Wikimedia Foundation generally makes contributed text available under
Creative Commons Attribution-ShareAlike 4.0 International (CC BY-SA 4.0),
subject to exceptions and additional requirements for imported or third-party
content. License: https://creativecommons.org/licenses/by-sa/4.0/

Attribution and contributor histories: https://pt.wikipedia.org/ (article
histories, when corresponding article identifiers are available).

Processing transformations may include extraction, Unicode normalization,
deduplication, tokenization and segmentation. These are not representations
of the original articles. The precise snapshot, upstream dataset provider,
article-level provenance, and applicable exceptions MUST be verified.

Intended Maia model uses: research, commercial use, public weights,
modification and redistribution. Whether and how CC BY-SA applies to
training, model weights and outputs requires a documented rights assessment.
No claim of unconditional permission or legal clearance is made here.

Frozen inventory SHA-256: `{INVENTORY_SHA256}`

Outstanding obligations: see `wikipedia_pt_evidence.json` and final
human-reviewed `license_registry.json` / `license_approval.json`.
'''
if NOTICE_PATH.exists():
    print('Existing NOTICE preserved:', NOTICE_PATH)
else:
    NOTICE_PATH.write_text(notice, encoding='utf-8')
    print('Created NOTICE draft:', NOTICE_PATH)

# Populate ONLY non-decision, factual pointers in the registry.
# Never change status, permitted uses, reviewer, or approval timestamps.
if not REGISTRY_PATH.exists():
    print('Registry does not exist yet; run the existing License Compliance Audit cell to create it, then rerun this cell.')
else:
    registry = json.loads(REGISTRY_PATH.read_text(encoding='utf-8'))
    assert registry.get('inventory_sha256') == INVENTORY_SHA256
    entry = registry['sources']['wikipedia_pt']
    changed = False
    for field, value in {
        'license_id': 'CC-BY-SA-4.0',
        'license_url': 'https://creativecommons.org/licenses/by-sa/4.0/',
    }.items():
        if not entry.get(field):
            entry[field] = value
            changed = True
    evidence_list = entry.setdefault('evidence_urls_or_paths', [])
    for ref in [str(EVIDENCE_PATH), str(NOTICE_PATH)] + [r['url'] for r in references]:
        if ref not in evidence_list:
            evidence_list.append(ref)
            changed = True
    if changed:
        # Save an independent backup before updating the registry.
        backup = REGISTRY_PATH.with_name('license_registry.before_wikimedia_review.json')
        if not backup.exists():
            backup.write_bytes(REGISTRY_PATH.read_bytes())
        REGISTRY_PATH.write_text(json.dumps(registry, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Registry updated with evidence references only; status:', entry.get('status'))

print('WIKIMEDIA RIGHTS REVIEW: REVIEW_REQUIRED')
print('Remaining wikipedia_pt chunks:', len(wiki_remaining))
print('Sampled records:', len(sample_examples))
print('Snapshot/third-party rights/human approval: NOT VERIFIED')
print('No corpus chunks, shards or SQLite records modified.')


## Wikipedia PT — incremental full-record provenance audit (read-only inputs)

Run this cell **after Wikimedia Rights Review**, before the license gate. It reads the 650 remaining gzip chunks in batches of **100 per execution** (adjust `PROVENANCE_CHUNKS_PER_RUN`), inspects **every JSONL record** in each selected chunk, and writes only new audit artifacts to `wikimedia_rights_review/provenance_audit/`. Re-run the cell to continue. It does **not** change the production database, input chunks, license registry, approval, or the production authorization switch.

It checks actual values of `dataset_id`, `dataset_revision`, `config`, `split`, `rights_status`, `eligible_for_training`, `source_url`, and `source_document_id`; records missing/inconsistent fields; produces a per-article URL manifest (one row per unique source URL per chunk), plus aggregated value counts and anomaly examples. It **does not** independently verify that dataset revisions are authentic upstream snapshots or resolve third-party licensing exceptions. All output is documentary evidence, not legal approval.

**Execution:** Run notebook setup and preflight cells, Wikimedia Rights Review, then this cell. **Stop before Production authorization** while the license audit is blocked. A new run processes the next unexamined chunks; if a run is interrupted, completed chunk reports are reused.


In [ ]:
# Wikipedia PT provenance audit — batch-resumable, read-only source data
import csv, gzip, json, hashlib, os, tempfile
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from urllib.parse import urlsplit

PROVENANCE_CHUNKS_PER_RUN = 100  # None = all remaining chunks (can take hours)
MAX_ANOMALY_EXAMPLES = 50
AUDIT_DIR = RIGHTS_DIR / 'provenance_audit'
CHUNK_REPORT_DIR = AUDIT_DIR / 'chunks'
CHUNK_REPORT_DIR.mkdir(parents=True, exist_ok=True)

# The source inventory and completion set come from the frozen notebook state.
assert INVENTORY_SHA256 == hashlib.sha256(json.dumps(
    config['selected_inventory'], sort_keys=True, ensure_ascii=False,
    separators=(',', ':')).encode('utf-8')).hexdigest()
assert len(wiki_remaining) == 650, 'Unexpected source inventory; inspect before continuing'
assert len({r['path'] for r in wiki_remaining}) == 650

# Atomic writes prevent partial reports from appearing completed after interruption.
def audit_atomic_json(path, payload):
    path = Path(path)
    tmp = path.with_name(path.name + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def audit_atomic_csv(path, rows, fieldnames):
    tmp = path.with_name(path.name + '.tmp')
    with tmp.open('w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

META_FIELDS = ('dataset_id','dataset_revision','config','split',
               'rights_status','eligible_for_training')
REQUIRED_FIELDS = META_FIELDS + ('source_url','source_document_id')

def value_key(v):
    if v is None:
        return '<NULL>'
    return json.dumps(v, sort_keys=True, ensure_ascii=False, default=str)[:300]

def is_valid_wikipedia_pt_url(url):
    if not isinstance(url, str):
        return False
    try:
        u = urlsplit(url)
        return (u.scheme == 'https' and u.hostname == 'pt.wikipedia.org'
                and u.path.startswith('/wiki/') and bool(u.path[6:])
                and not u.username and not u.password)
    except ValueError:
        return False

# No text content is persisted in the audit outputs.
def scan_wiki_chunk(entry):
    path = Path(entry['path'])
    assert path.is_file(), f'Missing source chunk: {path}'
    counts = {k: Counter() for k in META_FIELDS}
    issues = Counter()
    examples = []
    # Aggregate unique URL -> record count, document ids; bounded by distinct articles
    articles = {}
    records = 0
    try:
        with gzip.open(path, 'rt', encoding='utf-8') as f:
            for lineno, line in enumerate(f, 1):
                if not line.strip():
                    issues['blank_line'] += 1
                    continue
                records += 1
                try:
                    row = json.loads(line)
                    if not isinstance(row, dict):
                        raise ValueError('JSONL record is not an object')
                except (ValueError, json.JSONDecodeError) as e:
                    issues['invalid_json_record'] += 1
                    if len(examples) < MAX_ANOMALY_EXAMPLES:
                        examples.append({'line':lineno,'issue':'invalid_json_record','detail':str(e)[:160]})
                    continue
                for field in META_FIELDS:
                    counts[field][value_key(row.get(field))] += 1
                for field in REQUIRED_FIELDS:
                    if row.get(field) is None or row.get(field) == '':
                        issues['missing_'+field] += 1
                url = row.get('source_url')
                if not is_valid_wikipedia_pt_url(url):
                    issues['invalid_source_url'] += 1
                    if len(examples) < MAX_ANOMALY_EXAMPLES:
                        examples.append({'line':lineno,'issue':'invalid_source_url','value':str(url)[:200]})
                if row.get('eligible_for_training') is not True:
                    issues['not_marked_training_eligible'] += 1
                # Preserve raw rights_status values; do not equate metadata with legal permission.
                if isinstance(url, str) and url:
                    item = articles.setdefault(url, {'records':0,'source_document_ids':set()})
                    item['records'] += 1
                    docid = row.get('source_document_id')
                    if docid is not None:
                        item['source_document_ids'].add(str(docid))
    except (OSError, EOFError, UnicodeError) as e:
        raise RuntimeError(f'Unable to read {path}: {e}') from e
    for field in META_FIELDS:
        non_null = [k for k in counts[field] if k != '<NULL>']
        if len(non_null) > 1:
            issues['multiple_'+field+'_values_in_chunk'] += 1
    # Save per-chunk attribution CSV separately, then report JSON as completion marker.
    key = hashlib.sha256(str(path).encode('utf-8')).hexdigest()[:16]
    csv_name = f'{path.stem}.{key}.articles.csv'
    article_rows = [
        {'chunk':path.name,'source_url':url,'record_count':d['records'],
         'source_document_ids':json.dumps(sorted(d['source_document_ids']), ensure_ascii=False)}
        for url,d in sorted(articles.items())
    ]
    audit_atomic_csv(CHUNK_REPORT_DIR / csv_name, article_rows,
                     ['chunk','source_url','record_count','source_document_ids'])
    report = {
        'schema_version':1, 'inventory_sha256':INVENTORY_SHA256,
        'chunk_path':str(path), 'chunk_name':path.name,
        'chunk_size_bytes':path.stat().st_size, 'chunk_mtime_ns':path.stat().st_mtime_ns,
        'scanned_utc':datetime.now(timezone.utc).isoformat(),
        'records':records,'unique_source_urls':len(articles),
        'field_value_counts':{k:dict(v) for k,v in counts.items()},
        'issues':dict(issues),'anomaly_examples':examples,
        'article_manifest_file':csv_name,
        'scope_note':'Metadata checks only; not independent rights clearance.'
    }
    report_path = CHUNK_REPORT_DIR / f'{path.name}.{key}.report.json'
    audit_atomic_json(report_path, report)
    return report

def load_completed_reports():
    reports = {}
    for entry in wiki_remaining:
        path = Path(entry['path'])
        key = hashlib.sha256(str(path).encode('utf-8')).hexdigest()[:16]
        report_file = CHUNK_REPORT_DIR / f'{path.name}.{key}.report.json'
        if not report_file.is_file():
            continue
        r = json.loads(report_file.read_text(encoding='utf-8'))
        assert r.get('inventory_sha256') == INVENTORY_SHA256, 'Audit inventory mismatch'
        assert r.get('chunk_path') == str(path), 'Audit chunk path mismatch'
        assert (CHUNK_REPORT_DIR / r['article_manifest_file']).is_file(), 'Missing article manifest'
        stat = path.stat()
        assert r['chunk_size_bytes'] == stat.st_size and r['chunk_mtime_ns'] == stat.st_mtime_ns, (
            'Chunk changed after provenance scan', path)
        reports[str(path)] = r
    return reports

previous = load_completed_reports()
remaining = [e for e in wiki_remaining if e['path'] not in previous]
run_batch = remaining if PROVENANCE_CHUNKS_PER_RUN is None else remaining[:PROVENANCE_CHUNKS_PER_RUN]
print(f'PROVENANCE AUDIT: {len(previous)}/650 already scanned; scanning {len(run_batch)} this run')
for i, entry in enumerate(run_batch, 1):
    scan_wiki_chunk(entry)
    if i % 10 == 0 or i == len(run_batch):
        print(f'  This run: {i}/{len(run_batch)} chunks scanned')

reports = load_completed_reports()
aggregate = {field:Counter() for field in META_FIELDS}
issue_totals = Counter()
for r in reports.values():
    for field in META_FIELDS:
        aggregate[field].update(r['field_value_counts'][field])
    issue_totals.update(r['issues'])

# A per-chunk manifest remains available even for a partial audit; avoid
# creating a huge all-article in-memory list or making an unverified completeness claim.
summary = {
    'schema_version':1,
    'generated_utc':datetime.now(timezone.utc).isoformat(),
    'status':'COMPLETE_PENDING_HUMAN_REVIEW' if len(reports)==len(wiki_remaining)
             else 'PARTIAL_PENDING_HUMAN_REVIEW',
    'source':'wikipedia_pt', 'inventory_sha256':INVENTORY_SHA256,
    'total_remaining_chunks':len(wiki_remaining),
    'scanned_chunks':len(reports), 'unscanned_chunks':len(wiki_remaining)-len(reports),
    'scanned_records':sum(r['records'] for r in reports.values()),
    'unique_urls_sum_per_chunk':sum(r['unique_source_urls'] for r in reports.values()),
    'metadata_value_counts':{k:dict(v) for k,v in aggregate.items()},
    'issue_counts':dict(issue_totals),
    'article_manifest_location':str(CHUNK_REPORT_DIR),
    'snapshot_verified':False,'third_party_exceptions_cleared':False,
    'human_rights_approval':False,
    'note':'Actual source metadata values, not upstream snapshot verification or license approval.'
}
summary_path = AUDIT_DIR / 'provenance_summary.json'
audit_atomic_json(summary_path, summary)
print('\nPROVENANCE STATUS:', summary['status'])
print('Chunks:', summary['scanned_chunks'], '/', summary['total_remaining_chunks'])
print('Records:', summary['scanned_records'])
print('Issues:', summary['issue_counts'])
for field, values in summary['metadata_value_counts'].items():
    print(f'{field}: {values}')
print('Summary:', summary_path)
print('Article manifests:', CHUNK_REPORT_DIR)
print('LICENSE APPROVAL: NOT GRANTED; production remains blocked until independent review.')


## License compliance — source-level review and documented approval

The inventory and source provenance must be reviewed **before** any remaining chunks are processed. The audit is a conservative evidence gate, **not** a legal opinion and **not** an automatic license grant.

Run the following audit cell. On first execution it creates `license_registry.json` with every source marked `pending_review` and stops. Fill it only after examining the actual snapshot, source terms and evidence. Each source that still has unprocessed chunks needs an explicit decision, reviewer, review date, evidence references, and a scope including `maia_llm_training`.

The approval file is **not generated by the notebook**. An authorized reviewer must create `license_approval.json` with the audit's inventory digest, scope, reviewer, date, and `approved: true`. Keep the rights evidence and any attribution/redistribution obligations separately. Existing completed data is not retrospectively cleared by this gate.


In [ ]:
# 10. Source license review — NO automatic rights approval
import json, hashlib
from datetime import datetime, timezone
from pathlib import Path

LICENSE_REGISTRY = WORK_DIR / 'license_registry.json'
LICENSE_AUDIT = WORK_DIR / 'license_audit.json'
LICENSE_APPROVAL = WORK_DIR / 'license_approval.json'

# Canonical digest binds decisions to the full frozen selected inventory.
INVENTORY_SHA256 = hashlib.sha256(
    json.dumps(config['selected_inventory'], sort_keys=True, ensure_ascii=False,
               separators=(',', ':')).encode('utf-8')
).hexdigest()

remaining_by_source = {}
for source in SOURCE_NAMES:
    remaining_by_source[source] = sum(
        1 for p in selected[source]
        if str(p) not in completed_paths
    )

if not LICENSE_REGISTRY.exists():
    template = {
        'schema_version': 1,
        'inventory_sha256': INVENTORY_SHA256,
        'sources': {
            source: {
                'status': 'pending_review',
                'license_id': None,
                'snapshot_or_dataset_version': None,
                'license_url': None,
                'evidence_urls_or_paths': [],
                'permitted_uses': [],
                'attribution_and_sharealike_obligations': None,
                'reviewer': None,
                'reviewed_utc': None,
                'notes': None,
            } for source in SOURCE_NAMES
        }
    }
    with LICENSE_REGISTRY.open('x', encoding='utf-8') as f:
        json.dump(template, f, indent=2, ensure_ascii=False)
    print('Created review template:', LICENSE_REGISTRY)

registry = json.loads(LICENSE_REGISTRY.read_text(encoding='utf-8'))
assert registry.get('inventory_sha256') == INVENTORY_SHA256, 'Registry inventory mismatch'
assert registry.get('schema_version') == 1
assert set(registry.get('sources', {})) == set(SOURCE_NAMES), 'Registry sources mismatch'

source_findings = {}
for source, remaining in remaining_by_source.items():
    entry = registry['sources'][source]
    # A decision must be explicit and supported by real evidence.
    checks = {
        'approved_status': entry.get('status') == 'approved',
        'license_identified': bool(entry.get('license_id')),
        'snapshot_identified': bool(entry.get('snapshot_or_dataset_version')),
        'license_reference': bool(entry.get('license_url')),
        'evidence_documented': bool(entry.get('evidence_urls_or_paths')),
        'training_scope': 'maia_llm_training' in entry.get('permitted_uses', []),
        'obligations_documented': bool(entry.get('attribution_and_sharealike_obligations')),
        'reviewer_identified': bool(entry.get('reviewer')),
        'review_date_valid': False,
    }
    try:
        stamp = datetime.fromisoformat(entry.get('reviewed_utc') or '')
        checks['review_date_valid'] = stamp.tzinfo is not None
    except (TypeError, ValueError):
        pass
    source_findings[source] = {
        'remaining_chunks': remaining,
        'status': 'PASS' if remaining == 0 or all(checks.values()) else 'REVIEW_REQUIRED',
        'checks': checks,
    }

status = 'PASS' if all(v['status'] == 'PASS' for v in source_findings.values()) else 'BLOCKED'
audit_report = {
    'schema_version': 1,
    'generated_utc': datetime.now(timezone.utc).isoformat(),
    'status': status,
    'inventory_sha256': INVENTORY_SHA256,
    'registry_sha256': hashlib.sha256(LICENSE_REGISTRY.read_bytes()).hexdigest(),
    'scope': 'maia_llm_training',
    'remaining_chunks': sum(remaining_by_source.values()),
    'sources': source_findings,
    'note': 'Evidence completeness gate only; human legal/rights approval remains necessary.',
}
LICENSE_AUDIT.write_text(json.dumps(audit_report, indent=2, ensure_ascii=False), encoding='utf-8')
print('LICENSE AUDIT:', status)
for source, item in source_findings.items():
    print(f"  {source}: remaining={item['remaining_chunks']} status={item['status']}")
print('Inventory SHA256:', INVENTORY_SHA256)
print('Audit report:', LICENSE_AUDIT)
print('Approval file (reviewer-created):', LICENSE_APPROVAL)


In [ ]:
# 11. Production authorization — fail closed, never auto-approve
# An authorized reviewer must create license_approval.json manually.
# Required fields: approved, reviewer, approved_utc, permitted_use,
# inventory_sha256, registry_sha256.
RUN_PRODUCTION = False  # Explicit opt-in AFTER rights approval.

assert LICENSE_AUDIT.is_file(), 'Run the license audit first.'
audit = json.loads(LICENSE_AUDIT.read_text(encoding='utf-8'))
assert audit.get('status') == 'PASS', 'Rights audit blocked; review source registry.'
assert audit.get('inventory_sha256') == INVENTORY_SHA256
assert audit.get('registry_sha256') == hashlib.sha256(LICENSE_REGISTRY.read_bytes()).hexdigest(), 'Registry changed after audit'
assert LICENSE_APPROVAL.is_file(), 'Missing independently documented reviewer approval.'
approval = json.loads(LICENSE_APPROVAL.read_text(encoding='utf-8'))
assert approval.get('approved') is True, 'Rights approval not granted.'
assert approval.get('inventory_sha256') == INVENTORY_SHA256
assert approval.get('registry_sha256') == audit['registry_sha256']
assert approval.get('permitted_use') == 'maia_llm_training'
assert isinstance(approval.get('reviewer'), str) and approval['reviewer'].strip()
try:
    approved_at = datetime.fromisoformat(approval['approved_utc'])
    assert approved_at.tzinfo is not None
except (KeyError, ValueError, TypeError) as exc:
    raise AssertionError('Approval must include an offset-aware ISO 8601 approved_utc') from exc
assert RUN_PRODUCTION, 'Set RUN_PRODUCTION=True only when ready to process remaining chunks.'
print('Production authorized for this session with documented source review.')


In [ ]:
# Process only missing chunks. Original process_chunk() handles one chunk per transaction.
import time
progress_dir=WORK_DIR/'progress'
progress_dir.mkdir(parents=True,exist_ok=True)
started=time.perf_counter(); new_chunks=0; new_tokens=0
for source in SOURCE_NAMES:
    for idx,path in enumerate(selected[source],1):
        if MAX_NEW_CHUNKS_PER_SESSION is not None and new_chunks>=MAX_NEW_CHUNKS_PER_SESSION: break
        if con.execute('SELECT 1 FROM completed_chunk WHERE path=?',(str(path),)).fetchone(): continue
        t0=time.perf_counter()
        result=process_chunk(source,path)
        elapsed=time.perf_counter()-t0
        assert result.startswith('OK '),result
        new_chunks+=1
        chunk_tokens=con.execute('SELECT tokens FROM completed_chunk WHERE path=?',(str(path),)).fetchone()[0]
        new_tokens+=chunk_tokens
        completed=con.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0]
        if new_chunks==1 or new_chunks%PROGRESS_EVERY_NEW_CHUNKS==0:
            print(f'[{completed:,}/{EXPECTED_TOTAL_CHUNKS:,}] {source} {idx} | {result} | {elapsed:.1f}s',flush=True)
        report={'timestamp_utc':datetime.now(timezone.utc).isoformat(),'source':source,
                'path':str(path),'new_chunks_this_session':new_chunks,
                'new_tokens_this_session':new_tokens,'completed_chunks':completed,
                'remaining_chunks':EXPECTED_TOTAL_CHUNKS-completed,
                'working_db':str(DB),'seed_db':str(SEED_DB)}
        tmp=progress_dir/'latest.json.tmp';dest=progress_dir/'latest.json'
        tmp.write_text(json.dumps(report,indent=2),encoding='utf-8');os.replace(tmp,dest)
        if new_chunks%VERIFY_EVERY_NEW_CHUNKS==0:
            assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok'
    if MAX_NEW_CHUNKS_PER_SESSION is not None and new_chunks>=MAX_NEW_CHUNKS_PER_SESSION: break
completed=con.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0]
print('SESSION COMPLETE:',new_chunks,'new chunks;',completed,'/',EXPECTED_TOTAL_CHUNKS)
print('STATUS:','ALL_CHUNKS_DONE' if completed==EXPECTED_TOTAL_CHUNKS else 'PARTIAL — rerun notebook')

In [ ]:
# Compact final status and conservation checks (no full shard hash scan).
assert con.execute('PRAGMA quick_check').fetchone()[0]=='ok'
seen_count=con.execute('SELECT COUNT(*) FROM seen').fetchone()[0]
docs_count=con.execute('SELECT COALESCE(SUM(docs),0) FROM completed_chunk').fetchone()[0]
assert seen_count==docs_count,(seen_count,docs_count)
assert con.execute('SELECT COUNT(*) FROM recovery_provenance').fetchone()[0]==306
completed=con.execute('SELECT COUNT(*) FROM completed_chunk').fetchone()[0]
print('STATUS:', 'COMPLETE' if completed==EXPECTED_TOTAL_CHUNKS else 'RESUMABLE')
print('Completed:',completed,'/',EXPECTED_TOTAL_CHUNKS)
print('Seen:',seen_count,'Documents:',docs_count)
print('Working DB:',DB)
print('Original state.sqlite3 and audited v5 were not overwritten.')
con.close()

## Operational cautions

- **Single writer only.** Do not run the old notebook or another session simultaneously.
- The source folder is still under rights review; complete that review before enabling processing.
- If interrupted after shard rename but before SQLite commit, the next session intentionally stops on an orphan. Inspect it rather than deleting it automatically.
- The operational SQLite is stored on Google Drive and may be slower than local SSD; no unsafe bidirectional syncing is attempted.
- Once all 21,021 chunks are complete, perform the final integrity/conservation audit before using the corpus for training. This notebook does **not** train the model.
